# Modul 17: PyTorch-Tensoren, DataLoader und dichte Netze | Lösungen

## Überblick

Sie untersuchen PyTorch-Tensoren, NumPy-Konvertierung, Broadcasting und Autograd. Danach erstellen Sie TensorDataset- und DataLoader-Objekte, definieren ein dichtes nn.Module, schreiben eine vollständige Trainings- und Auswertungsschleife und speichern den Modellzustand reproduzierbar.

**Zugehörige Vorlesungen**

- **Tensoren und Loader**
- **Dichtes PyTorch-Netz**

## Lernziele

Nach der Bearbeitung können Sie:

- PyTorch-Tensoren mit passenden Formen, Datentypen und kontrollierter NumPy-Speicherfreigabe erzeugen.
- Gradienten mit requires_grad und backward berechnen sowie Daten reproduzierbar mit DataLoadern bereitstellen.
- ein dichtes PyTorch-Modell trainieren, zwischen train und eval wechseln und state_dict sicher speichern und laden.

## Geprüfte Fähigkeiten

- Tensorformen, dtype, clone, detach, Broadcasting und Autograd
- TensorDataset, random_split, DataLoader, Batching und Shuffling
- nn.Module, BCEWithLogitsLoss, Optimierer, Trainingsschleife, Metriken und state_dict

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** anspruchsvoll
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle lädt einen kleinen binären Tabellendatensatz, teilt ihn reproduzierbar in Training, Validierung und Test und skaliert die Merkmale ohne Leakage. Alle PyTorch-Berechnungen laufen standardmäßig auf der CPU.

In [ ]:
import copy
import os
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

geraet = torch.device("cpu")
print("PyTorch-Version:", torch.__version__)
print("Verwendetes Gerät:", geraet)

daten = load_breast_cancer()
X_gesamt = daten.data.astype("float32")
y_gesamt = daten.target.astype("float32")

X_train_roh, X_test_roh, y_train_np, y_test_np = train_test_split(
    X_gesamt,
    y_gesamt,
    test_size=0.20,
    stratify=y_gesamt,
    random_state=RANDOM_SEED,
)
X_train_roh, X_val_roh, y_train_np, y_val_np = train_test_split(
    X_train_roh,
    y_train_np,
    test_size=0.20,
    stratify=y_train_np,
    random_state=RANDOM_SEED,
)

scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train_roh).astype("float32")
X_val_np = scaler.transform(X_val_roh).astype("float32")
X_test_np = scaler.transform(X_test_roh).astype("float32")

print("Train, Validierung, Test:", X_train_np.shape, X_val_np.shape, X_test_np.shape)

### Aufgabe 1: Tensoren, Datentypen und NumPy-Speicher kontrollieren

Erzeugen Sie aus dem vorgegebenen NumPy-Array einmal einen Tensor mit `torch.from_numpy` und einmal mit `torch.tensor`. Ändern Sie anschließend einen Wert im NumPy-Array und prüfen Sie beide Tensoren.

Erklären Sie den Unterschied zwischen gemeinsam genutztem Speicher und einer Kopie. Demonstrieren Sie außerdem `clone`, `detach` und die Rückkonvertierung eines CPU-Tensors nach NumPy. Geben Sie Form, Rang und Datentyp aus.

In [ ]:
array_basis = np.array([[1.0, 2.0], [3.0, 4.0]], dtype=np.float32)

# ============================================================
# MUSTERLÖSUNG
# ============================================================

tensor_geteilt = torch.from_numpy(array_basis)
tensor_kopie = torch.tensor(array_basis)

print("Form:", tensor_geteilt.shape)
print("Rang:", tensor_geteilt.ndim)
print("Datentyp:", tensor_geteilt.dtype)

array_basis[0, 0] = 99.0
print("NumPy nach Änderung:\n", array_basis)
print("from_numpy teilt Speicher:\n", tensor_geteilt)
print("torch.tensor besitzt Kopie:\n", tensor_kopie)

# clone() erstellt neuen Tensorinhalt. detach() trennt einen Tensor vom Autograd-Graphen.
unabhaengig = tensor_geteilt.clone().detach()
array_basis[0, 1] = -5.0
print("Unabhängiger Clone nach weiterer NumPy-Änderung:\n", unabhaengig)

zurueck_numpy = unabhaengig.cpu().numpy()
print("Rückkonvertierter Typ:", type(zurueck_numpy).__name__)
assert np.array_equal(zurueck_numpy, unabhaengig.numpy())

> **Musterantwort und Interpretation**
>
> Geteilter Speicher vermeidet Kopierkosten und ist praktisch, wenn NumPy-Daten effizient als CPU-Tensoren verwendet werden sollen. Er ist riskant, wenn eine spätere Änderung in einem Objekt unbemerkt das andere verändert. Für reproduzierbare Zwischenergebnisse oder sichere Übergaben ist eine explizite Kopie mit tensor(), clone() oder copy() oft geeigneter.

### Aufgabe 2: Indexieren, Broadcasting und Autograd prüfen

Erzeugen Sie eine Variable `x` der Form `(3, 2)` mit `requires_grad=True` und einen Biasvektor `b` der Form `(2,)`, ebenfalls mit Gradienten. Berechnen Sie

`verlust = sum((x + b)^2)`

und rufen Sie `backward()` auf. Geben Sie die Gradientenformen und -werte aus. Leiten Sie den Biasgradienten manuell her und prüfen Sie ihn mit `torch.allclose`. Setzen Sie danach die Gradienten mit `zero_()` zurück.

In [ ]:
x_autograd = torch.tensor(
    [[1.0, -1.0], [2.0, 0.5], [-0.5, 3.0]],
    dtype=torch.float32,
    requires_grad=True,
)
b_autograd = torch.tensor([0.2, -0.3], dtype=torch.float32, requires_grad=True)

# ============================================================
# MUSTERLÖSUNG
# ============================================================

verschoben = x_autograd + b_autograd
verlust_autograd = torch.sum(verschoben**2)
verlust_autograd.backward()

print("Verlust:", float(verlust_autograd.item()))
print("x-Gradientenform:", x_autograd.grad.shape)
print("x-Gradienten:\n", x_autograd.grad)
print("b-Gradientenform:", b_autograd.grad.shape)
print("b-Gradienten:", b_autograd.grad)

# b wird über alle drei Zeilen gebroadcastet. Deshalb summiert sein Gradient
# die Beiträge 2*(x+b) entlang der Batchachse.
manueller_b_gradient = 2.0 * verschoben.detach().sum(dim=0)
print("Manueller b-Gradient:", manueller_b_gradient)
assert torch.allclose(b_autograd.grad, manueller_b_gradient)

with torch.no_grad():
    x_autograd.grad.zero_()
    b_autograd.grad.zero_()
print("Gradienten nach dem Zurücksetzen:", x_autograd.grad, b_autograd.grad)

> **Musterantwort und Interpretation**
>
> Derselbe Biaswert beeinflusst jede Zeile des Batches. Nach der Kettenregel erhält der Parameter deshalb einen Beitrag aus jedem Beispiel, das ihn verwendet. Autograd sammelt diese Beiträge automatisch entlang der gebroadcasteten Achse.

### Aufgabe 3: TensorDataset und DataLoader reproduzierbar erstellen

Wandeln Sie die drei Datensplits in Float32-Tensoren um. Die Merkmalsform soll `(N, 30)` und die Zielform `(N, 1)` sein. Erstellen Sie TensorDataset-Objekte und DataLoader mit Batchgröße 32.

Nur der Trainingsloader soll mischen. Verwenden Sie für das Training einen `torch.Generator` mit festem Seed. Inspizieren Sie den ersten Batch und prüfen Sie, dass Validierung und Test jede Beobachtung genau einmal enthalten.

In [ ]:
# Erzeugen Sie aus jedem Split ein TensorDataset und anschließend einen DataLoader.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def als_tensordataset(X, y):
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
    return TensorDataset(X_tensor, y_tensor)

train_dataset = als_tensordataset(X_train_np, y_train_np)
val_dataset = als_tensordataset(X_val_np, y_val_np)
test_dataset = als_tensordataset(X_test_np, y_test_np)

generator = torch.Generator().manual_seed(RANDOM_SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    generator=generator,
    num_workers=0,
)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

batch_X, batch_y = next(iter(train_loader))
print("Trainingsbatch Merkmale:", batch_X.shape, batch_X.dtype)
print("Trainingsbatch Ziele:", batch_y.shape, batch_y.dtype)
print("Batches pro Epoche:", len(train_loader))

anzahl_val = sum(len(labels) for _, labels in val_loader)
anzahl_test = sum(len(labels) for _, labels in test_loader)
print("Validierungsbeispiele:", anzahl_val)
print("Testbeispiele:", anzahl_test)
assert anzahl_val == len(val_dataset)
assert anzahl_test == len(test_dataset)

> **Musterantwort und Interpretation**
>
> Mischen verhindert, dass die ursprüngliche Reihenfolge immer dieselben Mini-Batch-Zusammensetzungen erzeugt. Das kann die Optimierung stabilisieren. Validierung und Test ändern keine Gewichte. Dort erleichtert eine feste Reihenfolge die Zuordnung von Vorhersagen, Labels und Originalbeispielen sowie reproduzierbare Fehleranalysen.

### Aufgabe 4: Ein dichtes nn.Module mit passenden Logits definieren

Definieren Sie eine Klasse `DichtesNetz`, die von `nn.Module` erbt. Verwenden Sie zwei verborgene Linear-Schichten mit höchstens 32 und 16 Einheiten, ReLU und optional Dropout. Die letzte Schicht soll genau einen Logit pro Beispiel ausgeben.

Prüfen Sie die Ausgabeform eines Batches. Berechnen Sie `BCEWithLogitsLoss` und erklären Sie, warum im Modell keine Sigmoid-Schicht benötigt wird.

In [ ]:
class DichtesNetz(nn.Module):
    def __init__(self, eingabe_merkmale, dropout_rate=0.10):
        super().__init__()
        pass

    def forward(self, x):
        pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

class DichtesNetz(nn.Module):
    def __init__(self, eingabe_merkmale, dropout_rate=0.10):
        super().__init__()
        # nn.Sequential macht den linearen Datenfluss kompakt sichtbar.
        self.netz = nn.Sequential(
            nn.Linear(eingabe_merkmale, 32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.netz(x)

torch.manual_seed(RANDOM_SEED)
dichtes_modell = DichtesNetz(eingabe_merkmale=X_train_np.shape[1], dropout_rate=0.10).to(geraet)
logits_demo = dichtes_modell(batch_X.to(geraet))
verlust_funktion = nn.BCEWithLogitsLoss()
verlust_demo = verlust_funktion(logits_demo, batch_y.to(geraet))

print(dichtes_modell)
print("Logitform:", logits_demo.shape)
print("Demoverlust:", float(verlust_demo.item()))
print("Trainierbare Parameter:", sum(p.numel() for p in dichtes_modell.parameters() if p.requires_grad))
assert logits_demo.shape == batch_y.shape

> **Musterantwort und Interpretation**
>
> Die kombinierte Implementierung arbeitet direkt mit unbeschränkten Logits und nutzt numerisch stabile Umformungen. Eine separat berechnete Sigmoid-Wahrscheinlichkeit kann bei sehr großen positiven oder negativen Logits auf exakt null oder eins runden und instabile Logarithmen erzeugen. Für Metriken oder Inferenz wird Sigmoid erst außerhalb der Verlustfunktion angewendet.

### Aufgabe 5: Trainings-, Validierungs- und Testschleifen schreiben

Schreiben Sie Funktionen für eine Trainingsepoche und für die Auswertung ohne Gradienten. Verwenden Sie Adam, Lernrate 0.001, `BCEWithLogitsLoss` und höchstens 35 Epochen. Speichern Sie Verlust, Accuracy und F1 für Training und Validierung.

Nutzen Sie Early Stopping mit Geduld 6 und speichern Sie im Arbeitsspeicher eine Kopie des besten `state_dict`. Stellen Sie diesen Zustand wieder her und bewerten Sie das Modell einmalig auf dem Testloader. Zeichnen Sie die Verlustkurven.

In [ ]:
def trainiere_epoche(modell, loader, optimizer, loss_fn, device):
    pass

def bewerte_modell(modell, loader, loss_fn, device):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def trainiere_epoche(modell, loader, optimizer, loss_fn, device):
    modell.train()
    verlust_summe = 0.0
    alle_labels = []
    alle_wahrscheinlichkeiten = []

    for merkmale, labels in loader:
        merkmale = merkmale.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = modell(merkmale)
        verlust = loss_fn(logits, labels)
        verlust.backward()
        optimizer.step()

        verlust_summe += verlust.item() * len(labels)
        alle_labels.append(labels.detach().cpu().numpy().ravel())
        alle_wahrscheinlichkeiten.append(torch.sigmoid(logits).detach().cpu().numpy().ravel())

    labels_np = np.concatenate(alle_labels)
    probs_np = np.concatenate(alle_wahrscheinlichkeiten)
    klassen_np = (probs_np >= 0.5).astype(int)
    return {
        "loss": verlust_summe / len(loader.dataset),
        "accuracy": accuracy_score(labels_np, klassen_np),
        "f1": f1_score(labels_np, klassen_np, zero_division=0),
    }


def bewerte_modell(modell, loader, loss_fn, device):
    modell.eval()
    verlust_summe = 0.0
    alle_labels = []
    alle_wahrscheinlichkeiten = []

    # no_grad() spart Speicher und verhindert unnötige Rechengraphen bei der Auswertung.
    with torch.no_grad():
        for merkmale, labels in loader:
            merkmale = merkmale.to(device)
            labels = labels.to(device)
            logits = modell(merkmale)
            verlust = loss_fn(logits, labels)
            verlust_summe += verlust.item() * len(labels)
            alle_labels.append(labels.cpu().numpy().ravel())
            alle_wahrscheinlichkeiten.append(torch.sigmoid(logits).cpu().numpy().ravel())

    labels_np = np.concatenate(alle_labels)
    probs_np = np.concatenate(alle_wahrscheinlichkeiten)
    klassen_np = (probs_np >= 0.5).astype(int)
    return {
        "loss": verlust_summe / len(loader.dataset),
        "accuracy": accuracy_score(labels_np, klassen_np),
        "f1": f1_score(labels_np, klassen_np, zero_division=0),
        "auc": roc_auc_score(labels_np, probs_np),
        "labels": labels_np,
        "probabilities": probs_np,
    }

torch.manual_seed(RANDOM_SEED)
dichtes_modell = DichtesNetz(X_train_np.shape[1], dropout_rate=0.10).to(geraet)
verlust_funktion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(dichtes_modell.parameters(), lr=0.001)

historie_torch = []
bester_val_verlust = np.inf
bester_zustand = None
geduld = 6
ohne_verbesserung = 0

for epoche in range(35):
    train_metriken = trainiere_epoche(dichtes_modell, train_loader, optimizer, verlust_funktion, geraet)
    val_metriken = bewerte_modell(dichtes_modell, val_loader, verlust_funktion, geraet)
    historie_torch.append(
        {
            "Epoche": epoche + 1,
            "train_loss": train_metriken["loss"],
            "val_loss": val_metriken["loss"],
            "train_f1": train_metriken["f1"],
            "val_f1": val_metriken["f1"],
        }
    )

    if val_metriken["loss"] < bester_val_verlust - 1e-5:
        bester_val_verlust = val_metriken["loss"]
        bester_zustand = copy.deepcopy(dichtes_modell.state_dict())
        ohne_verbesserung = 0
    else:
        ohne_verbesserung += 1

    if ohne_verbesserung >= geduld:
        print("Early Stopping nach Epoche", epoche + 1)
        break

dichtes_modell.load_state_dict(bester_zustand)
test_metriken = bewerte_modell(dichtes_modell, test_loader, verlust_funktion, geraet)
print(
    "Test: Loss={loss:.4f}, Accuracy={accuracy:.3f}, F1={f1:.3f}, AUC={auc:.3f}".format(
        **test_metriken
    )
)

historie_torch_df = pd.DataFrame(historie_torch)
plt.plot(historie_torch_df["Epoche"], historie_torch_df["train_loss"], label="Training")
plt.plot(historie_torch_df["Epoche"], historie_torch_df["val_loss"], label="Validierung")
plt.xlabel("Epoche")
plt.ylabel("BCEWithLogitsLoss")
plt.title("PyTorch-Trainingsverlauf")
plt.legend()
plt.show()

> **Musterantwort und Interpretation**
>
> train() aktiviert das Trainingsverhalten zustandsabhängiger Schichten wie Dropout und BatchNorm. eval() schaltet diese Schichten in den Inferenzmodus. no_grad() verhindert zusätzlich den Aufbau von Rechengraphen, spart Speicher und beschleunigt die Auswertung. eval() ersetzt no_grad() nicht, und no_grad() ersetzt eval() nicht.

### Aufgabe 6: Integrationsaufgabe: state_dict speichern, laden und prüfen

Speichern Sie den besten Modellzustand zusammen mit wesentlichen Metadaten in einem temporären Verzeichnis. Erzeugen Sie eine neue Modellinstanz, laden Sie den Zustand mit `map_location="cpu"` und setzen Sie das Modell in den Evaluationsmodus.

Prüfen Sie, ob die ersten zehn Logits und Wahrscheinlichkeiten vor und nach dem Laden übereinstimmen. Implementieren Sie außerdem eine Funktion `sichere_einzelvorhersage`, die Form, endliche Werte und Merkmalsanzahl prüft, die gespeicherte Skalierung anwendet und Wahrscheinlichkeit sowie Klasse zurückgibt.

In [ ]:
def sichere_einzelvorhersage(rohzeile, modell, scaler_objekt, erwartete_merkmale, schwelle=0.5):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def sichere_einzelvorhersage(rohzeile, modell, scaler_objekt, erwartete_merkmale, schwelle=0.5):
    """Validiert eine rohe Tabellenzeile und führt reproduzierbare CPU-Inferenz aus."""
    zeile = np.asarray(rohzeile, dtype=np.float32).reshape(1, -1)
    if zeile.shape[1] != erwartete_merkmale:
        raise ValueError(f"Erwartet {erwartete_merkmale} Merkmale, erhalten {zeile.shape[1]}.")
    if not np.isfinite(zeile).all():
        raise ValueError("Die Eingabe enthält NaN oder unendliche Werte.")

    skaliert = scaler_objekt.transform(zeile).astype("float32")
    tensor = torch.tensor(skaliert, dtype=torch.float32)
    modell.eval()
    with torch.no_grad():
        logit = modell(tensor).item()
        wahrscheinlichkeit = torch.sigmoid(torch.tensor(logit)).item()
    return {
        "Wahrscheinlichkeit": wahrscheinlichkeit,
        "Klasse": int(wahrscheinlichkeit >= schwelle),
        "Schwelle": schwelle,
    }

referenz_batch = torch.tensor(X_test_np[:10], dtype=torch.float32)
dichtes_modell.eval()
with torch.no_grad():
    referenz_logits = dichtes_modell(referenz_batch).clone()
    referenz_probs = torch.sigmoid(referenz_logits)

with tempfile.TemporaryDirectory() as temp_ordner:
    checkpoint_pfad = os.path.join(temp_ordner, "dichtes_netz.pt")
    checkpoint = {
        "state_dict": dichtes_modell.state_dict(),
        "eingabe_merkmale": X_train_np.shape[1],
        "dropout_rate": 0.10,
        "schwelle": 0.5,
        "torch_version": torch.__version__,
        "random_seed": RANDOM_SEED,
    }
    torch.save(checkpoint, checkpoint_pfad)

    geladen = torch.load(checkpoint_pfad, map_location="cpu", weights_only=False)
    neues_modell = DichtesNetz(
        eingabe_merkmale=geladen["eingabe_merkmale"],
        dropout_rate=geladen["dropout_rate"],
    )
    neues_modell.load_state_dict(geladen["state_dict"])
    neues_modell.eval()

    with torch.no_grad():
        neue_logits = neues_modell(referenz_batch)
        neue_probs = torch.sigmoid(neue_logits)

    print("Logits identisch:", torch.allclose(referenz_logits, neue_logits, atol=1e-7))
    print("Wahrscheinlichkeiten identisch:", torch.allclose(referenz_probs, neue_probs, atol=1e-7))
    assert torch.allclose(referenz_logits, neue_logits, atol=1e-7)

    beispiel_ergebnis = sichere_einzelvorhersage(
        X_test_roh[0],
        neues_modell,
        scaler,
        erwartete_merkmale=X_train_np.shape[1],
        schwelle=geladen["schwelle"],
    )
    print("Sichere Einzelvorhersage:", beispiel_ergebnis)

> **Musterantwort und Interpretation**
>
> state_dict enthält Parameterwerte, aber nicht automatisch die Python-Klassendefinition, Architekturparameter, Vorverarbeitung, Merkmalsreihenfolge, Schwelle oder Datenbeschreibung. Für reproduzierbare Inferenz müssen diese Bestandteile versioniert und gemeinsam dokumentiert werden. Beim Laden unbekannter Dateien ist außerdem Vorsicht geboten, weil Serialisierungsformate Sicherheitsrisiken haben können.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?